# ⚡ Calgary Household Energy Consumption
## End-to-End EDA + Regression Pipeline
> **Dataset:** [Kaggle — Calgary Household Energy Consumption (Synthetic)](https://www.kaggle.com/datasets/yonathanhary/household-energy-consumption-synthetic/data)  
> **Target:** `Daily_kWh` — predict daily electricity consumption  
> **Author:** Yonathan Hary Hutagalung

### Notebook Structure
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Modelling — 5 Baseline Models
5. Hyperparameter Tuning — XGBoost + GridSearchCV
6. Model Export

---
## 1. Setup & Data Loading

In [ ]:
# Install dependencies (uncomment if on Colab/Kaggle)
# !pip install xgboost seaborn plotly -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings, joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
print('Libraries loaded ✅')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
# Option A: Kaggle environment (dataset already mounted)
# df = pd.read_csv('/kaggle/input/household-energy-consumption-synthetic/calgary_household_energy_synthetic.csv')

# Option B: Download via Kaggle API
# !kaggle datasets download -d yonathanhary/household-energy-consumption-synthetic --unzip -p ./data

# Option C: Local file
df = pd.read_csv('calgary_household_energy_synthetic.csv', parse_dates=['Date'])

print(f'Shape: {df.shape}')
print(f'Date range: {df.Date.min()} → {df.Date.max()}')
df.head()

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── 2.1 Basic Info ────────────────────────────────────────────────────────────
print('=== Data Types & Non-Null Counts ===')
df.info()

In [ ]:
# ── 2.2 Descriptive Statistics ───────────────────────────────────────────────
df.describe().round(3)

In [ ]:
# ── 2.3 Missing Values ───────────────────────────────────────────────────────
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None ✅')

In [ ]:
# ── 2.4 Target Distribution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['Daily_kWh'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Daily kWh', fontsize=13)
axes[0].set_xlabel('Daily kWh'); axes[0].set_ylabel('Count')

axes[1].boxplot(df['Daily_kWh'], vert=False, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Boxplot — Daily kWh', fontsize=13)
axes[1].set_xlabel('Daily kWh')

plt.tight_layout(); plt.show()
print(f"Skewness: {df['Daily_kWh'].skew():.3f}")

In [ ]:
# ── 2.5 Monthly Average Consumption ──────────────────────────────────────────
df['Month'] = df['Date'].dt.month
monthly_avg = df.groupby('Month')['Daily_kWh'].mean()

plt.figure(figsize=(10, 4))
monthly_avg.plot(kind='bar', color='steelblue', edgecolor='white', alpha=0.85)
plt.title('Average Daily kWh by Month (all households)', fontsize=13)
plt.xlabel('Month'); plt.ylabel('Avg Daily kWh')
plt.xticks(ticks=range(12),
           labels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
           rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.6 Temperature vs kWh scatter ───────────────────────────────────────────
sample = df.sample(5000, random_state=42)

plt.figure(figsize=(10, 5))
scatter = plt.scatter(sample['Outside_Temperature_C'], sample['Daily_kWh'],
                      c=sample['Has_EV_Car'], cmap='coolwarm', alpha=0.4, s=15)
plt.colorbar(scatter, label='Has EV Car')
plt.title('Temperature vs Daily kWh (coloured by EV ownership)', fontsize=13)
plt.xlabel('Outside Temperature (°C)'); plt.ylabel('Daily kWh')
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.7 EV vs Non-EV Consumption ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
df.groupby('Has_EV_Car')['Daily_kWh'].plot(kind='density', ax=ax, legend=True)
ax.set_title('KDE: Daily kWh — EV vs Non-EV Households', fontsize=13)
ax.set_xlabel('Daily kWh')
ax.legend(['No EV', 'Has EV'])
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.8 Household Size vs Avg kWh ────────────────────────────────────────────
size_avg = df.groupby('Household_Size')['Daily_kWh'].mean()

plt.figure(figsize=(7, 4))
size_avg.plot(kind='bar', color='coral', edgecolor='white', alpha=0.85)
plt.title('Avg Daily kWh by Household Size', fontsize=13)
plt.xlabel('Household Size (people)'); plt.ylabel('Avg Daily kWh')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# ── 2.9 Correlation Heatmap ───────────────────────────────────────────────────
num_cols = ['Outside_Temperature_C','Household_Size','Living_Area_m2',
            'Has_EV_Car','Max_AC_Hours','AC_Hours_Used',
            'Tariff_Rate_CAD_kWh','Daily_kWh']

plt.figure(figsize=(10, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout(); plt.show()

---
## 3. Feature Engineering

In [ ]:
# ── Date Features ────────────────────────────────────────────────────────────
df['DayOfWeek']  = df['Date'].dt.dayofweek          # 0=Mon
df['IsWeekend']  = (df['DayOfWeek'] >= 5).astype(int)
df['Year']       = df['Date'].dt.year
# Month already created in EDA section above

# Season encoding
season_map = {12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}
df['Season_enc'] = df['Month'].map(season_map)

# Degree-day features (key for energy modelling)
df['HDD'] = np.maximum(0, 18 - df['Outside_Temperature_C'])  # Heating Degree Days
df['CDD'] = np.maximum(0, df['Outside_Temperature_C'] - 18)  # Cooling Degree Days

# Monthly sunlight hours (Calgary)
sunlight = {1:5.6,2:7.0,3:8.8,4:10.2,5:11.2,6:12.3,
            7:13.0,8:11.4,9:9.0,10:7.4,11:5.4,12:4.9}
df['Sunlight_Hours'] = df['Month'].map(sunlight)

print('New features added:', ['DayOfWeek','IsWeekend','Year','Season_enc','HDD','CDD','Sunlight_Hours'])
df.head(3)

In [ ]:
# ── Define Feature Matrix & Target ───────────────────────────────────────────
FEATURES = [
    'Outside_Temperature_C', 'Household_Size', 'Living_Area_m2',
    'Has_EV_Car', 'Max_AC_Hours', 'AC_Hours_Used',
    'Tariff_Rate_CAD_kWh', 'Month', 'DayOfWeek', 'IsWeekend',
    'Season_enc', 'HDD', 'CDD', 'Sunlight_Hours'
]
TARGET = 'Daily_kWh'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

---
## 4. Modelling — 5 Baseline Models

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    mae  = mean_absolute_error(y_te, preds)
    r2   = r2_score(y_te, preds)
    print(f'{name:<30} RMSE={rmse:.3f}  MAE={mae:.3f}  R²={r2:.4f}')
    return {'Model': name, 'RMSE': round(rmse,3), 'MAE': round(mae,3), 'R2': round(r2,4), 'fitted': model}

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

results = []

# 1. Linear Regression
results.append(evaluate('Linear Regression',
    LinearRegression(), X_train_sc, y_train, X_test_sc, y_test))

# 2. Ridge Regression
results.append(evaluate('Ridge Regression',
    Ridge(alpha=1.0), X_train_sc, y_train, X_test_sc, y_test))

# 3. Decision Tree
results.append(evaluate('Decision Tree',
    DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    X_train, y_train, X_test, y_test))

# 4. Random Forest
results.append(evaluate('Random Forest',
    RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    X_train, y_train, X_test, y_test))

# 5. XGBoost (baseline)
results.append(evaluate('XGBoost (baseline)',
    XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=RANDOM_STATE,
                 verbosity=0, n_jobs=-1),
    X_train, y_train, X_test, y_test))

In [ ]:
# ── Baseline Results Table ────────────────────────────────────────────────────
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'fitted'} for r in results])
results_df.sort_values('RMSE')

In [ ]:
# ── Visualise Baseline Comparison ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = ['RMSE', 'MAE', 'R2']
colors  = ['#e74c3c', '#f39c12', '#2ecc71']

for ax, metric, color in zip(axes, metrics, colors):
    ax.barh(results_df['Model'], results_df[metric], color=color, alpha=0.8, edgecolor='white')
    ax.set_title(metric, fontsize=13)
    ax.invert_yaxis()
    for i, v in enumerate(results_df[metric]):
        ax.text(v * 0.98, i, f'{v:.3f}', va='center', ha='right', fontsize=9, color='white', fontweight='bold')

plt.suptitle('Baseline Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5. Hyperparameter Tuning — XGBoost + GridSearchCV

In [ ]:
# ── GridSearchCV Parameter Grid ───────────────────────────────────────────────
param_grid = {
    'n_estimators':  [200, 400],
    'max_depth':     [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0],
}

xgb_base = XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1)

grid_search = GridSearchCV(
    xgb_base,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
print('Best params:', grid_search.best_params_)
print(f'Best CV RMSE: {-grid_search.best_score_:.3f}')

In [ ]:
# ── Evaluate Tuned Model ──────────────────────────────────────────────────────
best_model = grid_search.best_estimator_
tuned_preds = best_model.predict(X_test)

tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_preds))
tuned_mae  = mean_absolute_error(y_test, tuned_preds)
tuned_r2   = r2_score(y_test, tuned_preds)

print('=== Tuned XGBoost Results ===')
print(f'RMSE : {tuned_rmse:.3f}')
print(f'MAE  : {tuned_mae:.3f}')
print(f'R²   : {tuned_r2:.4f}')

In [ ]:
# ── Actual vs Predicted Plot ──────────────────────────────────────────────────
sample_idx = np.random.choice(len(y_test), size=500, replace=False)
y_sample    = np.array(y_test)[sample_idx]
pred_sample = tuned_preds[sample_idx]

plt.figure(figsize=(7, 6))
plt.scatter(y_sample, pred_sample, alpha=0.4, s=20, color='steelblue')
lims = [min(y_sample.min(), pred_sample.min()) - 1,
        max(y_sample.max(), pred_sample.max()) + 1]
plt.plot(lims, lims, 'r--', lw=1.5, label='Perfect Prediction')
plt.xlabel('Actual kWh'); plt.ylabel('Predicted kWh')
plt.title('Actual vs Predicted — Tuned XGBoost', fontsize=13)
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
feat_imp = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
feat_imp.plot(kind='barh', color='steelblue', alpha=0.85, edgecolor='white')
plt.title('Feature Importance — Tuned XGBoost', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout(); plt.show()

In [ ]:
# ── Residuals Plot ────────────────────────────────────────────────────────────
residuals = y_test - tuned_preds

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(tuned_preds, residuals, alpha=0.3, s=15, color='steelblue')
axes[0].axhline(0, color='red', lw=1.5, ls='--')
axes[0].set_title('Residuals vs Predicted', fontsize=12)
axes[0].set_xlabel('Predicted kWh'); axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].set_title('Residual Distribution', fontsize=12)
axes[1].set_xlabel('Residual (kWh)')

plt.tight_layout(); plt.show()

---
## 6. Model Export

In [ ]:
# ── Save best model for Streamlit app ────────────────────────────────────────
joblib.dump(best_model, 'best_model.sav')
print('Model saved → best_model.sav ✅')

# Verify it loads correctly
loaded = joblib.load('best_model.sav')
verify_preds = loaded.predict(X_test[:5])
print('Verification predictions:', verify_preds.round(3))
print('\nFeature order expected by Streamlit app:')
for i, f in enumerate(FEATURES, 1):
    print(f'  {i:2}. {f}')